# Data Cleaning Project
## Oasis Infobyte — Data Analytics Internship
### Name: Priti Ranjit | Track: Data Analytics | Task: L1 Task 3

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

All libraries imported successfully!


In [8]:
# Load the dataset
df = pd.read_csv('Housing.csv')

# First look
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape of dataset: (545, 13)

First 5 rows:


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## Step 1: Data Quality Report
Here we check the overall quality of the dataset before cleaning.

In [9]:
print("="*50)
print("DATA QUALITY REPORT")
print("="*50)

print("\n1. Dataset Shape:", df.shape)
print("\n2. Column Data Types:")
print(df.dtypes)

print("\n3. Missing Values per Column:")
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n4. Duplicate Rows:", df.duplicated().sum())

DATA QUALITY REPORT

1. Dataset Shape: (545, 13)

2. Column Data Types:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object

3. Missing Values per Column:
Empty DataFrame
Columns: [Missing Count, Missing Percentage]
Index: []

4. Duplicate Rows: 0


In [10]:
plt.figure(figsize=(12, 6))
missing_data = df.isnull().sum().sort_values(ascending=False)
missing_data = missing_data[missing_data > 0]

if len(missing_data) > 0:
    sns.barplot(x=missing_data.values, y=missing_data.index, 
                color='coral')
    plt.title('Missing Values per Column', fontsize=16)
    plt.xlabel('Number of Missing Values')
    plt.tight_layout()
    plt.savefig('missing_values.png')
    plt.show()
else:
    print("No missing values found!")

No missing values found!


<Figure size 1200x600 with 0 Axes>

## Step 2: Handle Missing Values
We use median for numbers and mode for text columns.

In [11]:
df_clean = df.copy()

# Numerical columns — fill with median
numerical_cols = df_clean.select_dtypes(
                 include=[np.number]).columns
for col in numerical_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val:.2f}")

# Categorical columns — fill with mode
categorical_cols = df_clean.select_dtypes(
                   include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

print("\nMissing values after cleaning:", 
      df_clean.isnull().sum().sum())


Missing values after cleaning: 0


C:\Users\user\AppData\Local\Temp\ipykernel_1508\3269559452.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_clean.select_dtypes(


## Step 3: Remove Duplicate Rows

In [12]:
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
after = len(df_clean)

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Duplicates removed: {before - after}")

Rows before: 545
Rows after: 545
Duplicates removed: 0


## Step 4: Fix Data Types

In [13]:
print("Data types before:")
print(df_clean.dtypes)

# Fix any object columns that should be numbers
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        try:
            df_clean[col] = pd.to_numeric(df_clean[col])
            print(f"Converted {col} to numeric")
        except:
            pass

print("\nData types after:")
print(df_clean.dtypes)

Data types before:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object

Data types after:
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object


## Step 5: Outlier Detection Using IQR Method

In [14]:
print("Outlier Analysis:")
print("="*50)

num_cols = df_clean.select_dtypes(include=[np.number]).columns

for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_clean[
        (df_clean[col] < lower) | 
        (df_clean[col] > upper)
    ]
    print(f"{col}: {len(outliers)} outliers detected")

Outlier Analysis:
price: 15 outliers detected
area: 12 outliers detected
bedrooms: 12 outliers detected
bathrooms: 1 outliers detected
stories: 41 outliers detected
parking: 12 outliers detected


## Step 6: Standardize Text Formatting

In [15]:
for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.strip()
        df_clean[col] = df_clean[col].str.title()

print("Text standardization complete!")
print("\nSample cleaned data:")
if len(categorical_cols) > 0:
    print(df_clean[categorical_cols[:3]].head())

Text standardization complete!

Sample cleaned data:
  mainroad guestroom basement
0      Yes        No       No
1      Yes        No       No
2      Yes        No      Yes
3      Yes        No      Yes
4      Yes       Yes      Yes


## Step 7: Before vs After Summary

In [16]:
summary = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Missing Values',
        'Duplicate Rows'
    ],
    'Before Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ],
    'After Cleaning': [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum()
    ]
})

print(summary.to_string(index=False))

        Metric  Before Cleaning  After Cleaning
    Total Rows              545             545
 Total Columns               13              13
Missing Values                0               0
Duplicate Rows                0               0


In [17]:
df_clean.to_csv('cleaned_dataset.csv', index=False)
print("Cleaned dataset saved!")
print(f"Final shape: {df_clean.shape}")

Cleaned dataset saved!
Final shape: (545, 13)


## Conclusion

1. Handled all missing values using median/mode imputation
2. Removed duplicate rows
3. Fixed incorrect data types
4. Detected outliers using IQR method
5. Standardized text formatting

**The dataset is now clean and ready for analysis!**